# Preparing and validating the analytical database

> **Historical v1 analysis.** The modelling target in this workflow ends in September 2025; newer supplemental observations remain separate here. For the current 519-game forecast, see [Notebook 20](20_fixed_cohort_2027_forecast.ipynb). [Notebook 17](17_fixed_sources_integration.ipynb) begins the separate v2 workflow.

![03 / Database Validation — Table relationships, stored observations and SQL checks](assets/pixel_theme/03_database_validation.svg)

The audit and screening identified software labels, missing metadata, and incomplete histories. This stage applies the working inclusion rule, stores the related records, and checks the category calculations before exploratory analysis.


In [1]:
from pathlib import Path
from contextlib import closing
import sqlite3
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
assert (ROOT / "src/prepare_data.py").exists(), "Open the project or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
from notebook_display import configure_notebook, present
configure_notebook()

from prepare_data import main

main()

Prepared 6,729 apps and 612,265 observations.
Eligible apps: 6,094; category/month rows: 795.


## 1. Apply the working cohort

An app qualifies when it has an Action, Adventure, Casual, Simulation, Strategy, RPG, Sports, or Racing label and no software/media flag. Mixed labels, software/media labels, inconclusive labels, and missing metadata remain held out from category aggregation.

Every tracked app is stored with its eligibility flag and reason, so the cohort can be reviewed without losing the underlying history. Since labels provide the screening evidence, genuine games with unclear or mixed assignments can remain outside the working sample.


In [2]:
games = pd.read_parquet(ROOT / "data/processed/games.parquet")
history = pd.read_parquet(ROOT / "data/processed/monthly_players.parquet")
labels = pd.read_parquet(ROOT / "data/processed/game_genres.parquet")
scope = pd.read_csv(ROOT / "reports/preparation_scope_impact.csv")
print(f"Tracked apps: {len(games):,}; eligible: {games['eligible'].sum():,}")
present(scope, 'Apply the working cohort')


Tracked apps: 6,729; eligible: 6,094


Review group,Eligible,Observed apps,Average concurrent activity,Activity share
Game-related labels only,1,"3,561","3,196,727.72",96.10%
Mixed game/software labels,0,9,"80,331.25",2.41%
Software/media labels only,0,84,"39,005.99",1.17%
Missing metadata,0,174,"7,491.31",0.23%
No decisive label,0,49,"2,953.35",0.09%


The latest-month impact table uses all recorded app activity that month as its denominator, making each group's contribution visible. App counts describe the breadth of the selected catalog, while activity share shows how much of the observed total it represents.


## 2. Store related records at a defined grain

The database separates one-app records in games, app/label assignments in game_genres, and app/month observations in monthly_players, with source_runs recording the input snapshots. These grains determine the keys used to identify observations and label assignments consistently.

Primary keys prevent duplicates at those grains, while foreign keys require referenced apps and sources to exist. The checks below compare imported row counts and inspect the database relationships and integrity.


In [3]:
with closing(sqlite3.connect(ROOT / "database/steam.sqlite")) as connection:
    counts = pd.DataFrame([
        {"table": table, "rows": connection.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]}
        for table in ["games", "game_genres", "monthly_players", "source_runs"]
    ])
    assert connection.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
    assert connection.execute("PRAGMA foreign_key_check").fetchall() == []
present(counts, 'Store related records at a defined grain')


Table,Rows
games,"6,729"
game_genres,"16,691"
monthly_players,"612,265"
source_runs,2


## 3. Preserve missing observations

Recorded zeros are retained, while missing app/month records remain absent from the observation table. The category query builds a continuous calendar and counts an app as expected within its own first-to-last recorded span.

For example, an app observed at zero in January and 12.5 in March has an unknown February observation. When no category apps are observed that month, activity remains SQL NULL, represented as NaN in pandas; when some are observed, the total covers those apps and the coverage columns show the shortfall.


## 4. Aggregate and check category activity

The SQL view connects eligible apps with their labels and monthly observations, then groups the result by category and month. Each game's full activity contributes to its applicable labels, so an Action and RPG game appears in both totals and adding them together would count its activity twice.


In [4]:
with closing(sqlite3.connect(ROOT / "database/steam.sqlite")) as connection:
    category = pd.read_sql_query("SELECT * FROM category_monthly", connection)
latest_month = category["month"].max()
present(category.loc[category["month"].eq(latest_month)].reset_index(drop=True), 'Aggregate and check category activity')


Category,Month,Average concurrent activity,Observed apps,Expected apps,Apps with internal gaps,Internal coverage
Action,Sep 2025,"2,503,728.38","1,606","1,606",0,100.00%
Indie,Sep 2025,"760,753.82","1,621","1,621",0,100.00%
Massively Multiplayer,Sep 2025,"693,909.44",236,236,0,100.00%
RPG,Sep 2025,"625,704.13",857,857,0,100.00%
Strategy,Sep 2025,"526,923.53","1,096","1,096",0,100.00%


The category avg_players value sums observed app-level monthly average concurrent activity, describing activity attributed to that label within the working cohort. Unique monthly users would require different information, while Indie and Massively Multiplayer describe different properties from gameplay labels such as Action or RPG.


In [5]:
# Reconcile SQL totals with an independently expressed pandas aggregation.
eligible_ids = games.loc[games["eligible"].eq(1), "app_id"]
selected_labels = labels.loc[labels["genre"].isin(category["category"].unique())]
joined = history.loc[history["app_id"].isin(eligible_ids)].merge(
    selected_labels, on="app_id", how="inner", validate="many_to_many"
)
expected = joined.groupby(["genre", "month"])["avg_players"].agg(["sum", "size"])
actual = category.set_index(["category", "month"])
pd.testing.assert_series_equal(
    actual["avg_players"], expected["sum"].reindex(actual.index),
    check_names=False, rtol=1e-10, atol=1e-6
)
pd.testing.assert_series_equal(
    actual["observed_apps"], expected["size"].reindex(actual.index, fill_value=0),
    check_names=False, check_dtype=False
)
assert not history.duplicated(["app_id", "month"]).any()
assert len(history) == 612_265
assert (category["observed_apps"] <= category["expected_apps"]).all()
print("SQL and pandas totals agree; observation keys, row counts, and coverage checks passed.")
present(category.loc[category["coverage_rate"].idxmin()].to_frame("lowest internal coverage"), 'Aggregate and check category activity')


SQL and pandas totals agree; observation keys, row counts, and coverage checks passed.


,Lowest internal coverage
category,Indie
month,Sep 2021
Average concurrent activity,"697,078.25"
Observed apps,"1,992"
Expected apps,"2,585"
Apps with internal gaps,593
Internal coverage,77.06%


## 5. Interpret internal coverage

Coverage divides observed eligible apps by eligible apps whose recorded spans include the month, measuring observation completeness within those spans. Periods before first observation and after last observation are outside the denominator.

At the latest month, a span can reach the endpoint only when the app has an observation there, which explains the resulting 100% coverage. Apps that disappeared earlier fall outside that denominator, leaving changes in the tracked catalog and whole-Steam representation to be assessed separately.

The later metadata snapshot is applied retrospectively to historical activity, so category totals need to be read alongside this label assumption, changing sample composition, and player behaviour.


## Preparation and validation findings

![Checkpoint 2 — findings, interpretation and the next decision](assets/pixel_theme/checkpoint_02.svg)

The prepared snapshot retains **6,729 apps and 612,265 observations**, with **6,094 apps eligible** under the working rule. Five overlapping categories produce **795 category/month rows** across July 2012–September 2025, with the database checks and SQL/pandas comparison supporting further exploration.

Recorded evidence: [prepared category data](../reports/category_monthly_prepared.csv).
